In [1]:
import numpy as np
import pandas as pd
import matplotlib
import warnings

In [ ]:
%matplotlib inline          
import matplotlib.pyplot as plt

In [3]:
warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

In [4]:
hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
scores = np.array([42, 48, 55, 59, 66, 70, 78, 81, 89, 91])

In [5]:
X = hours.reshape(-1, 1)        
y = scores

In [6]:
X_grid = np.linspace(0.5, 10.5, 300).reshape(-1, 1)

In [7]:
df = pd.DataFrame({"Student": [f"S{i}" for i in range(1, 11)],
                   "Study Hours (x)": hours, "Exam Score (y)": scores})

In [8]:
print("DATASET\n")
print(df.to_string(index=False))
print("\nDescriptive statistics:\n", df[["Study Hours (x)", "Exam Score (y)"]].describe())
print("\nPearson correlation x-y:", round(np.corrcoef(hours, scores)[0, 1], 4))

DATASET

Student  Study Hours (x)  Exam Score (y)
     S1                1              42
     S2                2              48
     S3                3              55
     S4                4              59
     S5                5              66
     S6                6              70
     S7                7              78
     S8                8              81
     S9                9              89
    S10               10              91

Descriptive statistics:
        Study Hours (x)  Exam Score (y)
count          10.0000         10.0000
mean            5.5000         67.9000
std             3.0277         16.9079
min             1.0000         42.0000
25%             3.2500         56.0000
50%             5.5000         68.0000
75%             7.7500         80.2500
max            10.0000         91.0000

Pearson correlation x-y: 0.9973


In [9]:
plt.scatter(X, y)
plt.xlabel("X")
plt.ylabel("y")
plt.title("X vs y")
plt.show()

In [10]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

In [11]:
LOO = LeaveOneOut()

In [12]:
# MSE
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [13]:
def train_metrics(model):
    # for the all model 
    model.fit(X, y)
    pred = model.predict(X) 
    return rmse(y, pred), r2_score(y, pred) # Return prediction error and R²

In [14]:
def loocv_rmse(model): # Performance using LOOCV
    neg_mse = cross_val_score(
        model, X, y,
        cv=LOO,
        scoring="neg_mean_squared_error"
    )
    return float(np.sqrt(-neg_mse.mean()))

In [15]:
def poly_pipeline(degree, regressor=None):
    if regressor is None: # (Linear Regression, Ridge, Lasso)
        regressor = LinearRegression()
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)), # creates new features like X² X³
        ("scale", StandardScaler()),
        ("reg", regressor),
    ])

In [16]:
# PART A

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(hours, scores, color="navy", s=60, zorder=3)
for i, (a, b) in enumerate(zip(hours, scores), 1):
    plt.annotate(f"S{i}", (a, b), textcoords="offset points", xytext=(5, -12), fontsize=8)
plt.xlabel("Study hours (x)")
plt.ylabel("Exam score (y)")
plt.title("Exam score vs study hours")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fig_A1_scatter.png", dpi=150)
plt.show()

In [20]:
models_A = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression (degree 2)": Pipeline([("poly", PolynomialFeatures(2, include_bias=False)),
                                                  ("reg", LinearRegression())]),
    "Polynomial Regression (degree 3)": Pipeline([("poly", PolynomialFeatures(3, include_bias=False)),
                                                  ("reg", LinearRegression())]),
}

In [ ]:
rows = []
plt.figure(figsize=(8, 5.5))
plt.scatter(hours, scores, color="black", s=60, zorder=5, label="Observations")
for (name, m), c in zip(models_A.items(), ["tab:blue", "tab:orange", "tab:green"]):
    tr_rmse, tr_r2 = train_metrics(m)
    rows.append([name, tr_rmse, tr_r2])
    plt.plot(X_grid, m.predict(X_grid), color=c, lw=2, label=name)
plt.xlabel("Study hours (x)")
plt.ylabel("Exam score (y)")
plt.title("Part A - Linear vs polynomial fits")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fig_A3_models.png", dpi=150)
plt.show()

In [24]:
tableA = pd.DataFrame(rows, columns=["Model", "Training RMSE", "Training R2"])
print(tableA.to_string(index=False))
tableA.to_csv("table_A.csv", index=False)

                           Model  Training RMSE  Training R2
               Linear Regression         1.1672       0.9947
Polynomial Regression (degree 2)         1.0435       0.9958
Polynomial Regression (degree 3)         1.0211       0.9959


In [25]:
lin = models_A["Linear Regression"]
print(f"Linear model: score = {lin.intercept_:.3f} + {lin.coef_[0]:.3f} * hours")

Linear model: score = 37.267 + 5.570 * hours


In [26]:
# PART B

In [27]:
degrees = [1, 2, 3, 5, 9]
rows = []
for d in degrees:
    m = poly_pipeline(d)
    tr_rmse, tr_r2 = train_metrics(m)
    rows.append([d, tr_rmse, loocv_rmse(m)])

In [28]:
tableB = pd.DataFrame(rows, columns=["Polynomial Degree", "Training RMSE", "LOOCV RMSE"])
print(tableB.to_string(index=False))
tableB.to_csv("table_B.csv", index=False)

 Polynomial Degree  Training RMSE  LOOCV RMSE
                 1         1.1672      1.5486
                 2         1.0435      1.5728
                 3         1.0211      2.0620
                 5         0.8943      3.7070
                 9         0.0000    222.1629


In [29]:
best_deg = int(tableB.loc[tableB["LOOCV RMSE"].idxmin(), "Polynomial Degree"])
print(f"\nLowest LOOCV RMSE -> degree {best_deg}")


Lowest LOOCV RMSE -> degree 1


In [30]:
best_poly_deg = int(tableB[tableB["Polynomial Degree"] >= 2]
                    .sort_values("LOOCV RMSE").iloc[0]["Polynomial Degree"])
print(f"Best polynomial with degree >= 2 -> degree {best_poly_deg}")

Best polynomial with degree >= 2 -> degree 2


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(tableB["Polynomial Degree"], tableB["Training RMSE"], "o--", label="Training RMSE")
plt.plot(tableB["Polynomial Degree"], tableB["LOOCV RMSE"], "s-", color="crimson", label="LOOCV RMSE")
plt.yscale("log")
plt.xlabel("Polynomial degree")
plt.ylabel("RMSE (log scale)")
plt.title("Part B - Complexity vs error")
plt.xticks(degrees)
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("fig_B_loocv_vs_degree.png", dpi=150)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["mathtext.fontset"] = "dejavusans"
plt.rcParams["text.usetex"] = False
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for ax, d in zip(axes, degrees):
    m = poly_pipeline(d).fit(X, y)
    ax.scatter(hours, scores, color="black", zorder=3)
    ax.plot(X_grid, m.predict(X_grid), color="tab:red")
    ax.set_title(f"Degree {d}")
    ax.set_xlabel("Study hours")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Exam score")
plt.tight_layout()
plt.savefig("fig_B_curves_by_degree.png", dpi=130)
plt.show()

In [35]:
# PART C

In [36]:
reps = {"Original: x": 1, "Polynomial: x, x^2, x^3": 3, "Higher-degree: x ... x^5": 5}
rows = [[k, loocv_rmse(poly_pipeline(d))] for k, d in reps.items()]
tableC1 = pd.DataFrame(rows, columns=["Representation", "LOOCV RMSE"])
print(tableC1.to_string(index=False))

          Representation  LOOCV RMSE
             Original: x      1.5486
 Polynomial: x, x^2, x^3      2.0620
Higher-degree: x ... x^5      3.7070


In [37]:
alphas = [0.01, 0.1, 1, 10]
rows = []

In [38]:
for name, Reg in [("Ridge", Ridge), ("Lasso", Lasso)]:
    for a in alphas:
        kwargs = {"max_iter": 100000} if name == "Lasso" else {}
        m = poly_pipeline(5, Reg(alpha=a, **kwargs))
        rows.append([name, a, loocv_rmse(m)])

In [39]:
tableC2 = pd.DataFrame(rows, columns=["Model", "alpha", "LOOCV RMSE"])
print("\nRegularization strength search (degree 5):")
print(tableC2.to_string(index=False))
tableC2.to_csv("table_C_alpha_search.csv", index=False)


Regularization strength search (degree 5):
Model   alpha  LOOCV RMSE
Ridge  0.0100      2.1442
Ridge  0.1000      1.8071
Ridge  1.0000      3.5735
Ridge 10.0000      7.2820
Lasso  0.0100      1.7654
Lasso  0.1000      1.6223
Lasso  1.0000      1.9187
Lasso 10.0000     11.8773


In [40]:
best_ridge = tableC2[tableC2.Model == "Ridge"].sort_values("LOOCV RMSE").iloc[0]
best_lasso = tableC2[tableC2.Model == "Lasso"].sort_values("LOOCV RMSE").iloc[0]
print(f"\nBest Ridge alpha = {best_ridge['alpha']}, best Lasso alpha = {best_lasso['alpha']}")


Best Ridge alpha = 0.1, best Lasso alpha = 0.1


In [ ]:
plt.figure(figsize=(7, 5))
for name, mk in [("Ridge", "o"), ("Lasso", "s")]:
    sub = tableC2[tableC2.Model == name]
    plt.plot(sub["alpha"], sub["LOOCV RMSE"], marker=mk, label=name)
plt.xscale("log")
plt.xlabel("alpha (log scale)")
plt.ylabel("LOOCV RMSE")
plt.title("Part C - Effect of regularization strength (degree 5)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fig_C_alpha_search.png", dpi=150)
plt.show()

In [43]:
tableC3 = pd.DataFrame([
    ["Linear Regression", 1, "None", loocv_rmse(poly_pipeline(1))],
    ["Polynomial Regression", 3, "None", loocv_rmse(poly_pipeline(3))],
    ["Polynomial Regression", 5, "None", loocv_rmse(poly_pipeline(5))],
    [f"Ridge Regression (alpha={best_ridge['alpha']})", 5, "L2", best_ridge["LOOCV RMSE"]],
    [f"Lasso Regression (alpha={best_lasso['alpha']})", 5, "L1", best_lasso["LOOCV RMSE"]],
], columns=["Model", "Feature Degree", "Regularization", "LOOCV RMSE"])

In [44]:
print("\nTable C:")
print(tableC3.to_string(index=False))
tableC3.to_csv("table_C.csv", index=False)


Table C:
                       Model  Feature Degree Regularization  LOOCV RMSE
           Linear Regression               1           None      1.5486
       Polynomial Regression               3           None      2.0620
       Polynomial Regression               5           None      3.7070
Ridge Regression (alpha=0.1)               5             L2      1.8071
Lasso Regression (alpha=0.1)               5             L1      1.6223


In [45]:
lasso_fit = poly_pipeline(5, Lasso(alpha=best_lasso["alpha"], max_iter=100000)).fit(X, y)
ridge_fit = poly_pipeline(5, Ridge(alpha=best_ridge["alpha"])).fit(X, y)
ols5_fit = poly_pipeline(5).fit(X, y)

In [46]:
coefs = pd.DataFrame({
    "OLS deg5": ols5_fit.named_steps["reg"].coef_,
    "Ridge": ridge_fit.named_steps["reg"].coef_,
    "Lasso": lasso_fit.named_steps["reg"].coef_,
}, index=["x", "x^2", "x^3", "x^4", "x^5"])
print("\nCoefficients (on standardized features):")
print(coefs)


Coefficients (on standardized features):
     OLS deg5   Ridge   Lasso
x     10.8081 12.9243 16.2906
x^2   56.7159  4.9912  0.0000
x^3 -170.5888  1.0901 -0.0000
x^4  204.0317 -1.0102 -0.0000
x^5  -85.2997 -2.4319 -0.4680


In [47]:
# PART D

In [48]:
candidates = {
    "Linear Regression": poly_pipeline(1),
    f"Polynomial Regression (deg {best_poly_deg})": poly_pipeline(best_poly_deg),
    f"Ridge (deg 5, alpha={best_ridge['alpha']})": poly_pipeline(5, Ridge(alpha=best_ridge["alpha"])),
    f"Lasso (deg 5, alpha={best_lasso['alpha']})": poly_pipeline(
        5, Lasso(alpha=best_lasso["alpha"], max_iter=100000)),
}

In [49]:
extra = {}

In [50]:
best = min([2, 3, 4], key=lambda k: loocv_rmse(DecisionTreeRegressor(max_depth=k, random_state=0)))
extra[f"Decision Tree (max_depth={best})"] = DecisionTreeRegressor(max_depth=best, random_state=0)

In [51]:
extra["Random Forest (200 trees)"] = RandomForestRegressor(n_estimators=200, random_state=0)

In [52]:
best_svr = min([(k, C) for k in ["linear", "rbf"] for C in [1, 10, 100, 1000]],
               key=lambda kc: loocv_rmse(Pipeline([("scale", StandardScaler()),
                                                   ("reg", SVR(kernel=kc[0], C=kc[1]))])))

In [53]:
extra[f"SVR (kernel={best_svr[0]}, C={best_svr[1]})"] = Pipeline(
    [("scale", StandardScaler()), ("reg", SVR(kernel=best_svr[0], C=best_svr[1]))])